# YOLOv8 Baseline Pipeline — 4 Classes CCTV

**Classes:** 0: fall | 1: fire_smoke | 2: ppe | 3: fight

**Workflow:**
1. Setup environment & check GPU
2. Load pretrained model (yolov8n)
3. Quick inference sanity check
4. Prepare dataset config (data.yaml)
5. Train (fine-tune)
6. Validate & evaluate
7. Inference with trained model
8. Export to ONNX

## 1. Setup Environment

In [ ]:
%pip install -q ultralytics roboflow

import torch
from ultralytics import YOLO

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — training will be slow!")

## 2. Load Pretrained Model

เริ่มจาก yolov8n (nano) — เร็วสุด เหมาะกับ real-time CCTV
ถ้าต้องการ accuracy สูงขึ้นเปลี่ยนเป็น yolov8s.pt

In [ ]:
MODEL_NAME = "yolov8n.pt"   # options: yolov8n.pt / yolov8s.pt

model = YOLO(MODEL_NAME)    # auto-download pretrained COCO weights
model.info()

## 3. Quick Inference Sanity Check

In [ ]:
# ทดสอบ inference กับรูปตัวอย่าง (bus.jpg มากับ repo)
results = model.predict(
    source="https://ultralytics.com/images/bus.jpg",
    conf=0.25,
    save=True,
)

for r in results:
    print(f"Detected {len(r.boxes)} objects")
    for box in r.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        print(f"  class={model.names[cls_id]}, conf={conf:.2f}, xyxy={box.xyxy[0].tolist()}")

## 4. Dataset Preparation

โครงสร้างที่ต้องการ:

    dataset/
    ├── images/{train,val,test}/
    └── labels/{train,val,test}/

In [ ]:
import os
from pathlib import Path

DATASET_ROOT = Path("../dataset")   # << แก้ path ให้ตรงกับที่เก็บ dataset

# ตรวจสอบโครงสร้าง dataset
for split in ["train", "val", "test"]:
    img_dir = DATASET_ROOT / "images" / split
    lbl_dir = DATASET_ROOT / "labels" / split
    n_img = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
    print(f"{split:5s}: images={n_img:5d} | labels={n_lbl:5d}")

# นับ instances ต่อ class (เช็ค class imbalance)
CLASS_NAMES = ['fall', 'fire_smoke', 'ppe', 'fight']
counts = {i: 0 for i in range(4)}
for lbl_file in (DATASET_ROOT / "labels").rglob("*.txt"):
    for line in lbl_file.read_text().splitlines():
        if line.strip():
            counts[int(line.split()[0])] += 1

print("\nInstances per class:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i}: {name:12s} -> {counts[i]}")

In [ ]:
# สร้าง data.yaml อัตโนมัติ
data_yaml_content = f"""
path: {DATASET_ROOT.resolve()}
train: images/train
val: images/val
test: images/test

nc: 4
names: ['fall', 'fire_smoke', 'ppe', 'fight']
"""

DATA_YAML = Path("data.yaml")
DATA_YAML.write_text(data_yaml_content.strip())
print(DATA_YAML.read_text())

## 5. Training (Fine-tune)

In [ ]:
EPOCHS = 100
IMGSZ = 640
BATCH = 16

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=30,          # early stopping
    save=True,
    plots=True,           # confusion matrix, PR curve ฯลฯ
    # augmentations
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
)

## 6. Validation & Evaluation

In [ ]:
# โหลด best weights จากการเทรน
best_model = YOLO("runs/detect/train/weights/best.pt")

metrics = best_model.val(data=str(DATA_YAML), split="val")

print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"mAP50    : {metrics.box.map50:.4f}")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:12s}: AP50={metrics.box.ap50[i]:.4f}")

## 7. Inference with Trained Model

In [ ]:
# ทดสอบกับรูป/วิดีโอ test set
TEST_SOURCE = str(DATASET_ROOT / "images" / "test")   # << แก้เป็น path รูป/วิดีโอที่ต้องการ

pred_results = best_model.predict(
    source=TEST_SOURCE,
    conf=0.25,
    iou=0.45,
    save=True,
    stream=True,
)

for r in pred_results:
    print(f"{Path(r.path).name}: {[best_model.names[int(c)] for c in r.boxes.cls]}")

## 8. Export for Deployment

In [ ]:
# Export เป็น ONNX สำหรับ deploy บน edge/server
onnx_path = best_model.export(format="onnx", imgsz=IMGSZ, dynamic=True)
print(f"Exported to: {onnx_path}")

# ตัวเลือกอื่น: format=torchscript / engine (TensorRT) / openvino